# 05_03 - Tail Risk Model: Upper-Tail Exceedances and Stress Days
## 1. Methodology Overview

This notebook focuses on the upper tail of the forecast distribution and on the days where realized prices exceed the model's risk signal.
The relevant project sources are `src/models/evaluate_model.py`, `src/pipeline/run_backtest.py`, and the backtest summaries under `data/outputs/backtests/`.

**Source artifacts used in this notebook:**
- `data/outputs/backtests/quantile_coverage_summary.csv`
- `data/outputs/backtests/quantile_upper_tail_exceedance_summary.csv`
- `data/outputs/backtests/extreme_days_vs_spot_only.csv`

The purpose is to read the real tail-risk outputs generated by the project's pipeline and inspect the days that matter most for procurement resilience.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

backtests_dir = Path('../../data/outputs/backtests')
coverage_path = backtests_dir / 'quantile_coverage_summary.csv'
tail_path = backtests_dir / 'quantile_upper_tail_exceedance_summary.csv'
extreme_days_path = backtests_dir / 'extreme_days_vs_spot_only.csv'

coverage_df = pd.read_csv(coverage_path)
tail_df = pd.read_csv(tail_path)
extreme_days_df = pd.read_csv(extreme_days_path)

print(f'Loaded coverage summary from: {coverage_path}')
print(f'Loaded upper-tail summary from: {tail_path}')
print(f'Loaded extreme-day comparison from: {extreme_days_path}')

display(coverage_df)
display(tail_df)
display(extreme_days_df.head())

## 2. Recomputing Tail Exceedances From the Real Validation Data

To keep the notebook fully reproducible, we also recompute the upper-tail exceedance days directly from the project's validation split and quantile model wrapper.
This uses the same code path as the pipeline, not a synthetic example.

In [ ]:
from src.models.evaluate_model import combine_quantile_predictions
from src.models.train_model import train_quantile_suite

train_df = pd.read_csv('../../data/processed/train_features.csv')
validation_df = pd.read_csv('../../data/processed/validation_features.csv')
quantile_output = train_quantile_suite(train_df=train_df, eval_df=validation_df)
prediction_frame = combine_quantile_predictions(quantile_output.results)
prediction_frame['q90_exceedance'] = prediction_frame['y_true'] > prediction_frame['q_0.9']

exceedance_count = int(prediction_frame['q90_exceedance'].sum())
exceedance_share = exceedance_count / len(prediction_frame)
print(f'Upper-tail exceedances: {exceedance_count} of {len(prediction_frame)} validation rows')
print(f'Exceedance share: {exceedance_share:.3%}')

display(prediction_frame.loc[prediction_frame['q90_exceedance']].sort_values('y_true', ascending=False).head(10))

## 3. Interpretation

The tail-risk model is not meant to replace the quantile model; it is a diagnostic layer that tells us when the upper tail is systematically breached and where the policy should be most cautious.
Those stress days are the ones the backtesting layer later compares across strategies.